# Tutorial: Geração de Dados Sintéticos de Qualidade
### Caso IPT — Análise Comportamental de Usuários em Interfaces Digitais

**Objetivo do tutorial:** ao final deste caderno, você deverá ser capaz de transformar sinais reais, hipóteses explícitas e escolhas metodológicas em uma base sintética defensável para análise.

Este notebook foi reorganizado para estudo autônomo. A ideia é controlar melhor a carga cognitiva: primeiro entendemos o problema, depois separamos evidência de hipótese, depois escolhemos a estrutura da base e só então codificamos a geração.

**Mapa do que vamos fazer:**

| Etapa | O que fazemos | Por que isso importa |
|-------|---------------|----------------------|
| 0 | Configuração do ambiente | Garantir reprodutibilidade |
| 1 | Entender a situação real | Saber o que já temos |
| 2A | Identificar lacunas | Separar observação de ausência de dado |
| 2B | Formular hipóteses | Transformar lacunas em decisões de modelagem |
| 3 | Definir unidade de análise | Fixar a granularidade da base |
| 4 | Montar o schema | Planejar as variáveis antes do código |
| 5 | Escolher distribuições | Usar uma família de distribuição por vez |
| 6 | Construir regras de geração | Traduzir hipóteses em funções |
| 7 | Gerar a base | Produzir o dataset sintético |
| 8 | Inserir outliers plausíveis | Tornar a base mais realista |
| 9 | Validar e explorar | Verificar se a base faz sentido |
| 10 | Exercícios práticos | Consolidar o raciocínio |

**Nota metodológica:** dados sintéticos não são dados inventados sem critério. Eles são dados gerados a partir de evidências parciais, hipóteses explícitas e regras documentadas.

## Etapa 0 — Configuração do ambiente

Vamos usar apenas bibliotecas comuns em notebooks introdutórios.

Fixar a semente aleatória é importante porque geração de dados envolve sorteios. Se todos usam a mesma semente, todos conseguem reproduzir os mesmos resultados e discutir as mesmas saídas.

In [1]:
import math
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ipt_utils import carregar_relatorios_sharepoint, serial_excel_para_data, exibir_cartoes, exibir_tabela_html

np.random.seed(42)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


def encontrar_arquivo_dados(nome_arquivo: str) -> Path:
    candidatos = [
        Path(nome_arquivo),
        Path('dados') / nome_arquivo,
        Path('../../dados') / nome_arquivo,
    ]
    for caminho in candidatos:
        if caminho.exists():
            return caminho.resolve()
    raise FileNotFoundError(f"Arquivo não encontrado: {nome_arquivo}")


print('Ambiente configurado com sucesso.')
print(f"   numpy  {np.__version__}")
print(f"   pandas {pd.__version__}")

Ambiente configurado com sucesso.
   numpy  2.4.4
   pandas 3.0.2


## Etapa 1 — A situação real: o que já sabemos

Antes de criar qualquer dado, vamos fazer um recorte deliberado do problema para não avançar mais do que os alunos precisam nesta primeira leitura.


### O caso IPT

O IPT (Instituto de Pesquisas Tecnológicas) quer tomar decisões sobre o comportamento de navegação na sua intranet, mas os dados disponíveis são **agregados e incompletos**.

**O que o parceiro quer:**
- Reduzir o tempo gasto procurando links operacionais
- Aumentar o engajamento com comunicados internos
- Segmentar usuários por perfil de comportamento
- Sustentar decisões de redesign e testes A/B

Neste notebook, a Etapa 1 vai trabalhar **apenas um desses objetivos**: `aumentar o engajamento com comunicados internos`.
Os demais ficam como espaço para expansão posterior dos grupos.


**O que os dados reais já mostram sobre esse objetivo** (fatos observados, sem inventar comportamento por sessão):


Objetivo deste bloco de código:
- usar os dados enviados pelo parceiro para fazer uma leitura descritiva de um único objetivo de negócio;
- identificar o que já conseguimos enxergar sobre comunicados internos antes de pensar em dados sintéticos.


Entrada:
- planilha `SiteAnalyticsData_24-abr.,2026.xlsx`;
- abas de tráfego geral, conteúdo popular, uso por dispositivo e uso por tempo.


Saída esperada:
- alguns indicadores-chave em Markdown;
- um ou mais gráficos `plotly` que ajudem a interpretar o objetivo escolhido.

### Estrutura da planilha recebida do IPT

A planilha `SiteAnalyticsData_24-abr.,2026.xlsx` contém 4 abas com dados agregados
— não há registro de sessão individual, login ou jornada de cliques.

| Aba | O que contém | Granularidade |
|---|---|---|
| Tráfego geral | Visualizadores exclusivos e visitas ao site por dia | 1 linha = 1 dia |
| Conteúdo popular | Páginas e posts mais visitados nos últimos 7 dias | 1 linha = 1 item de conteúdo |
| Uso por dispositivo | Acessos divididos por desktop, mobile app, mobile web, tablet | 1 linha = 1 dia |
| Uso por tempo | Média de viewers por dia da semana e faixa horária | 1 linha = 1 janela (dia × hora) |

O que **não** existe na planilha: perfil do usuário, tempo de leitura por sessão,
sequência de navegação, resultado de cada visita.

In [2]:
caminho_dados = encontrar_arquivo_dados('SiteAnalyticsData_24-abr.,2026.xlsx')
relatorios_sharepoint = carregar_relatorios_sharepoint(caminho_dados)

trafego_geral = relatorios_sharepoint['Tráfego geral'].iloc[11:].copy()
trafego_geral.columns = ['data_serial', 'visualizadores_exclusivos', 'visitas_ao_site', 'coluna_extra']
trafego_geral['data_serial'] = pd.to_numeric(trafego_geral['data_serial'], errors='coerce')
trafego_geral = trafego_geral.dropna(subset=['data_serial']).copy()
trafego_geral['data'] = trafego_geral['data_serial'].apply(serial_excel_para_data)
trafego_geral['visualizadores_exclusivos'] = pd.to_numeric(
    trafego_geral['visualizadores_exclusivos'], errors='coerce'
)
trafego_geral['visitas_ao_site'] = pd.to_numeric(trafego_geral['visitas_ao_site'], errors='coerce')
trafego_geral['dia_tipo'] = np.where(
    trafego_geral['data'].dt.dayofweek < 5, 'Dia útil', 'Fim de semana'
)

conteudo_popular = relatorios_sharepoint['Conteúdo popular'].iloc[4:].copy()
conteudo_popular.columns = [
    'conteudo',
    'tipo_conteudo',
    'visualizadores_exclusivos_7d',
    'visitas_7d',
    'coluna_extra',
]
conteudo_popular = conteudo_popular[conteudo_popular['conteudo'].notna()].copy()
conteudo_popular = conteudo_popular[
    ~conteudo_popular['conteudo'].astype(str).str.startswith('Clique aqui')
].copy()
conteudo_popular['visualizadores_exclusivos_7d'] = pd.to_numeric(
    conteudo_popular['visualizadores_exclusivos_7d'], errors='coerce'
)
conteudo_popular['visitas_7d'] = pd.to_numeric(conteudo_popular['visitas_7d'], errors='coerce')

uso_dispositivo = relatorios_sharepoint['Uso por dispositivo'].iloc[4:].copy()
uso_dispositivo.columns = [
    'data_serial',
    'desktop',
    'mobile_app',
    'mobile_web',
    'tablet',
    'outros',
]
uso_dispositivo['data_serial'] = pd.to_numeric(uso_dispositivo['data_serial'], errors='coerce')
uso_dispositivo = uso_dispositivo.dropna(subset=['data_serial']).copy()
for coluna in ['desktop', 'mobile_app', 'mobile_web', 'tablet', 'outros']:
    uso_dispositivo[coluna] = pd.to_numeric(uso_dispositivo[coluna], errors='coerce').fillna(0)

uso_tempo = relatorios_sharepoint['Uso por tempo'].iloc[5:].copy()
uso_tempo.columns = [
    'janela_original',
    'media_viewers_7d',
    'media_viewers_30d',
    'media_viewers_90d',
]
uso_tempo = uso_tempo[uso_tempo['janela_original'].notna()].copy()
padrao_dias = '^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday) '
uso_tempo = uso_tempo[uso_tempo['janela_original'].astype(str).str.match(padrao_dias)].copy()
uso_tempo['media_viewers_7d'] = pd.to_numeric(uso_tempo['media_viewers_7d'], errors='coerce').fillna(0)
uso_tempo['dia_semana_en'] = uso_tempo['janela_original'].str.extract('^(\\w+)')
uso_tempo['hora_inicio'] = uso_tempo['janela_original'].apply(
    lambda texto: datetime.strptime(texto.split(' ', 1)[1].split(' - ')[0], '%I %p').hour
)

mapa_dias = {
    'Sunday': 'Dom',
    'Monday': 'Seg',
    'Tuesday': 'Ter',
    'Wednesday': 'Qua',
    'Thursday': 'Qui',
    'Friday': 'Sex',
    'Saturday': 'Sab',
}
ordem_dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
uso_tempo['dia_semana'] = pd.Categorical(
    uso_tempo['dia_semana_en'].map(mapa_dias),
    categories=ordem_dias,
    ordered=True,
)

objetivo_negocio_focal = 'Aumentar o engajamento com comunicados internos'
comunicados = conteudo_popular[conteudo_popular['tipo_conteudo'] == 'News Post'].copy()
comunicados_top = comunicados.nlargest(8, 'visitas_7d').copy()
comunicados_top['conteudo_curto'] = comunicados_top['conteudo'].str.replace('.aspx', '', regex=False)
comunicados_top['conteudo_curto'] = comunicados_top['conteudo_curto'].str.replace('-', ' ', regex=False)
comunicados_top['conteudo_curto'] = comunicados_top['conteudo_curto'].str.slice(0, 40)

desktop_total = uso_dispositivo[['desktop', 'mobile_app', 'mobile_web', 'tablet', 'outros']].sum().sum()
percentual_desktop = uso_dispositivo['desktop'].sum() / desktop_total

pico_7d = uso_tempo.sort_values('media_viewers_7d', ascending=False).iloc[0]
sinais_reais = pd.DataFrame(
    [
        ['objetivo_focal', objetivo_negocio_focal, 'Recorte didático desta etapa'],
        ['visitas_news_post_7d', int(comunicados['visitas_7d'].sum()), 'Soma das visitas dos comunicados listados'],
        ['viewers_news_post_7d', int(comunicados['visualizadores_exclusivos_7d'].sum()), 'Visualizadores únicos dos comunicados listados'],
        ['percentual_desktop', round(percentual_desktop, 3), 'Participação do desktop no tráfego observado'],
        ['pico_uso', pico_7d['janela_original'], 'Maior média de visualizadores no relatorio de uso por tempo'],
    ],
    columns=['sinal', 'valor', 'observacao'],
)

### O que os dados já mostram sobre comunicados internos

| Indicador | Valor | Fonte |
|---|---|---|
| Objetivo trabalhado nesta etapa | Engajamento com comunicados | Recorte único para preservar espaço de expansão |
| Visitas aos comunicados (últimos 7 dias) | 1.894 | Aba "Conteúdo popular", itens do tipo News Post |
| Share de desktop (últimos 90 dias) | 97,4% | Aba "Uso por dispositivo" |
| Pico de atenção no site | Qua, 8h–9h | Janela com maior média de viewers no recorte de 7 dias |

In [3]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'xy'}, {'type': 'domain'}]],
    subplot_titles=(
        'Comunicados com mais visitas nos últimos 7 dias',
        'Participação dos dispositivos no tráfego observado',
    ),
)

fig.add_trace(
    go.Bar(
        x=comunicados_top['visitas_7d'],
        y=comunicados_top['conteudo_curto'],
        orientation='h',
        marker_color='#1d4ed8',
        text=comunicados_top['visitas_7d'],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Visitas 7d: %{x}<extra></extra>',
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Pie(
        labels=['Desktop', 'Mobile app', 'Mobile web', 'Tablet', 'Outros'],
        values=uso_dispositivo[['desktop', 'mobile_app', 'mobile_web', 'tablet', 'outros']].sum().tolist(),
        hole=0.55,
        marker_colors=['#0f172a', '#38bdf8', '#60a5fa', '#cbd5e1', '#94a3b8'],
        textinfo='percent+label',
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=520,
    width=1100,
    title_text='Etapa 1: o que os dados reais deixam ver sobre comunicados internos',
    showlegend=False,
    margin=dict(l=20, r=20, t=80, b=20),
)
fig.update_xaxes(title_text='Visitas nos últimos 7 dias', row=1, col=1)
fig.update_yaxes(title_text='', row=1, col=1)
fig.show()

matriz_tempo = (
    uso_tempo.pivot_table(
        index='dia_semana',
        columns='hora_inicio',
        values='media_viewers_7d',
        aggfunc='mean',
    )
    .reindex(ordem_dias)
    .fillna(0)
)

fig_heatmap = go.Figure(
    data=[
        go.Heatmap(
            z=matriz_tempo.values,
            x=[f"{hora:02d}h" for hora in matriz_tempo.columns],
            y=matriz_tempo.index.tolist(),
            colorscale=[
                [0.0, '#eff6ff'],
                [0.35, '#93c5fd'],
                [0.7, '#2563eb'],
                [1.0, '#0f172a'],
            ],
            colorbar_title='Viewers médios<br>7d',
            hovertemplate='Dia: %{y}<br>Hora inicial: %{x}<br>Média viewers 7d: %{z:.0f}<extra></extra>',
        )
    ]
)
fig_heatmap.update_layout(
    height=420,
    width=1100,
    title_text='Contexto temporal do site: quando é mais provável que um comunicado seja visto',
    xaxis_title='Hora inicial da faixa',
    yaxis_title='Dia da semana',
    margin=dict(l=20, r=20, t=70, b=20),
)
fig_heatmap.show()

### Reflita antes de continuar

Olhe para os indicadores e para os gráficos acima e responda mentalmente:

1. O que já conseguimos dizer sobre **exposição** aos comunicados?
2. O que ainda não conseguimos dizer sobre **leitura real** ou **resultado** desses comunicados?
3. Qual gráfico parece mais útil para abrir a conversa com os alunos: conteúdo, dispositivo ou tempo?

<details>
<summary>Possível direção de resposta</summary>

Esperamos algo como:

- já vemos quais comunicados concentram visitas, em que janelas de tempo o site fica mais movimentado e que o acesso é quase todo desktop;
- ainda não vemos tempo de leitura, percurso de navegação, perfil do usuário nem efeito do comunicado sobre alguma ação posterior;
- a combinação entre ranking de comunicados e heatmap horário abre uma boa conversa sobre o que é evidenciado e o que ainda precisa ser hipotetizado.

</details>


## Etapa 2A — Identificando lacunas

Nesta etapa, vamos usar o objetivo escolhido para deixar mais visivel a diferenca entre **o que o dado mostra** e **o que o caso ainda pede para entender**.


A pergunta aqui é simples:

> O que o caso exige que a gente entenda, mas os dados atuais ainda não entregam diretamente?

Ainda não vamos decidir distribuição, nem escrever função. Primeiro mapeamos o que existe e o que falta.

Objetivo desta etapa:
- organizar as lacunas principais do objetivo de engajamento com comunicados;
- deixar explícito o que é evidência e o que ainda exigirá hipótese.

Entrada:
- sinais reais lidos da planilha;
- recorte de negocio definido na Etapa 1.


Saída esperada:
- uma tabela curta em Markdown com as lacunas mais importantes do objetivo focal.

### Mapa de lacunas para o objetivo de engajamento com comunicados

| Dimensão | O que já vemos | O que ainda não vemos | Cobertura | Ponte para hipótese |
|---|---|---|---|---|
| Interesse por conteúdo | Ranking de News Posts por visitas e viewers nos últimos 7 dias | Se a visita virou leitura efetiva ou apenas clique rápido | ✅ Forte | Modelar tempo de leitura e abandono rápido por sessão |
| Momento de exposição | Pico claro de uso na quarta de manhã, concentração entre 8h e 11h | Horário específico de cada comunicado e efeito da publicação | ⚠️ Parcial | Usar pico horário como âncora geral, não como verdade por conteúdo |
| Perfil de quem engaja | O parceiro fala em perfis, mas a planilha não segmenta viewers | Quem são os leitores mais recorrentes de comunicados | 🔴 Crítica | Criar perfis sintéticos com pesos e preferências diferentes |
| Jornada até o comunicado | Sabemos quais itens foram visitados, mas não a sequência | Se o usuário veio da home, da busca ou de um atalho | 🔴 Crítica | Representar a sessão com origem, pageviews e uso de busca |
| Resultado após ler | Temos visitas e viewers, mas não há sinal de ação posterior | Se o comunicado levou a uma tarefa ou consulta adicional | 🔴 Crítica | Adicionar variável de sucesso dependente do objetivo |

## Etapa 2B — Transformando lacunas em hipóteses

Agora sim aparece a inferência. Em vez de deixar a lacuna abstrata, vamos transformar **uma delas** em uma hipótese de modelagem explícita.

Exemplo guiado para o objetivo focal:

| Lacuna observada | Hipótese defensável |
|---|---|
| Não sabemos quem lê os comunicados nem como essa leitura se distribui por sessão. | Em dias úteis, a maior parte das sessões de leitura de comunicados ocorre em desktop, entre 8h e 11h, com perfis `Administrativo` e `Operacional` mais presentes do que os demais. |

Repare que a hipótese:

- reaproveita o que os dados reais sugerem (`desktop`, pico matinal, uso em dias úteis);
- completa o que a planilha não entrega (perfil, sessão, recorrência);
- deixa claro onde termina a evidência e onde começa a arbitração modelada.

Abordagem incorreta comum:

- misturar o que foi observado com o que foi suposto;
- inventar proporções sem declarar que são hipóteses;
- tratar a hipótese como se fosse dado real.

Consequência:
- a base fica difícil de justificar;
- a análise exploratória parece precisa demais, mas não é transparente.

### Mini-exercício 2.1

Expanda o raciocínio para **outro objetivo do parceiro**. Por exemplo: `reduzir o tempo gasto procurando links operacionais`.

Tente escrever, em uma frase, a dupla:

- qual é a lacuna observada;
- qual seria uma hipótese sintética plausível para começar a modelagem.

<details>
<summary>Possível direção de resposta</summary>

Uma resposta plausível seria:

- lacuna: a planilha mostra muito tráfego em horários de expediente, mas não mostra quais sessões foram de busca operacional nem quanto tempo foi gasto até o clique útil;
- hipótese: sessões com objetivo operacional usam mais busca interna, têm mais pageviews e concentram maior tempo de busca antes do sucesso.

O importante aqui não é a frase exata, mas a passagem clara de `lacuna observada` para `hipótese explícita`.

</details>

Agora que separamos observação de hipótese, precisamos decidir em que granularidade a base será construída.


## Etapa 3 — Definindo a unidade de análise

Antes de escrever qualquer gerador, precisamos responder: **cada linha do dataset vai representar o quê?**

Essa decisão define tudo: quais variáveis fazem sentido, quais métricas são possíveis e até que tipo de gráfico descritivo caberia depois.


### Quais as opções?

### O que cada unidade de análise nos deixaria mostrar

| Pergunta analítica | Usuário | Sessão | Página visitada | Clique |
|---|:---:|:---:|:---:|:---:|
| Comparar perfis acumulados | ● | ◑ | ○ | ○ |
| Comparar tempo de busca e leitura | ○ | ● | ◑ | ○ |
| Ver picos por dia e hora | ○ | ● | ◑ | ◑ |
| Entender quais comunicados atraem visitas | ○ | ◑ | ● | ◑ |
| Reconstruir o caminho detalhado de navegação | ○ | ◑ | ◑ | ● |

● forte   ◑ parcial   ○ não se aplica

### Decisão metodológica adotada neste tutorial

**Unidade escolhida: sessão**

A sessão preserva o contexto mínimo para combinar objetivo, tempo, uso de busca,
pageviews e sucesso — sem empurrar para o nível de detalhe de clique.

- **Por que não clique:** detalhe demais; aumenta volume e complexidade logo no primeiro contato.
- **Por que não usuário:** contexto de menos; esconde variáveis situacionais como horário e abandono.
- **Tamanho inicial:** 5.000 sessões — suficiente para explorar distribuições e cruzamentos.

Uma transição importante: depois de mapear as lacunas, precisamos escolher como os dados serão representados. Essa decisão aparece na unidade de análise.

Objetivo desta etapa:
- comparar o que muda quando a linha da base representa usuário, sessão, página visitada ou clique;
- registrar a escolha metodológica adotada no tutorial.

Entrada:
- perguntas analiticas que queremos responder no AF01;
- objetivo de negocio e lacunas identificadas nas etapas anteriores.


Saída esperada:
- uma tabela comparativa entre unidades de análise;
- a decisão metodológica do notebook registrada antes da geração da base.

In [4]:
decisoes_metodologicas = {
    'unidade_de_analise': 'sessao',
    'justificativa': (
        'A sessão preserva o contexto mínimo para combinar objetivo, tempo, uso de busca, '
        'pageviews e sucesso sem empurrar a turma para o nível de detalhe de clique.'
    ),
    'n_sessoes': 5000,
    'periodo_representado': 'dias_uteis_principalmente',
    'seed': 42,
}

### Mini-exercício 3.1

Sem alterar o código: se o grupo escolhesse `usuário` em vez de `sessão` como unidade de análise, quais variáveis deixariam de fazer sentido do jeito que estão hoje?

<details>
<summary>Possível direção de resposta</summary>

Esperamos que você perceba que variáveis fortemente situacionais, como `faixa_horaria`, `objetivo_sessao` e `abandono_rapido`, estão naturalmente associadas à sessão, não ao usuário agregado ao longo do tempo.

A resposta não precisa ser única, mas deve mostrar que você entendeu que granularidade altera o significado das variáveis.

</details>

Erro comum:

- escolher a unidade mais detalhada possível só porque parece mais sofisticada.

No nosso caso, mais detalhe não significa necessariamente melhor aprendizagem ou melhor aderência ao AF01.

## Etapa 4 — Montando o schema da base

O schema é o "contrato" da base sintética. Ele documenta cada variável antes de gerá-la, forçando você a pensar sobre:

- Como essa variável vai ser gerada?
- Qual o seu tipo estatístico?
- O que ela representa no problema real?

>  **Dica prática:** montar o schema *antes* de escrever o código de geração evita a armadilha de gerar primeiro e interpretar depois — que é quando os dados sintéticos ficam sem sentido.

Agora que a granularidade está definida, precisamos estruturar as variáveis que cada linha vai carregar. Isso nos leva ao schema da base.

Objetivo desta etapa:
- declarar as variáveis da base antes de gerá-las;
- explicitar tipo de dado, nível de mensuração e estratégia de geração.

Entrada:
- unidade de análise definida como sessão;
- hipóteses sobre o problema do IPT.

Saída esperada:
- um schema documentado que já ajuda diretamente o AF01.

### Schema da base sintética — sessão como unidade de análise

| Variável | Tipo de dado | Nível de mensuração | Estratégia de geração | Descrição |
|---|---|---|---|---|
| `session_id` | identificador | nominal | sequencial | Identificador único da sessão |
| `perfil_usuario` | categórica | nominal | amostragem por pesos | Perfil comportamental do usuário |
| `tipo_dispositivo` | categórica | nominal | proporções observadas | Desktop, mobile web ou app |
| `dia_semana` | categórica | nominal | pesos por dia | Dia da semana da sessão |
| `faixa_horaria` | categórica | ordinal | pesos por faixa | Bloco de horário de acesso |
| `origem_acesso` | categórica | nominal | regra condicional | Como o usuário chegou à intranet |
| `objetivo_sessao` | categórica | nominal | condicional por perfil | Intenção principal da sessão |
| `pageviews` | quantitativa discreta | razão | Poisson(λ por objetivo) | Páginas visitadas na sessão |
| `tempo_busca_segundos` | quantitativa contínua | razão | log-normal por objetivo | Tempo gasto procurando |
| `tempo_leitura_segundos` | quantitativa contínua | razão | log-normal por objetivo | Tempo gasto lendo/consumindo |
| `tempo_total_segundos` | quantitativa contínua | razão | soma de componentes | Duração total da sessão |
| `usou_busca` | binária | nominal | Bernoulli condicional | 1 se usou a ferramenta de busca |
| `clicou_comunicado` | binária | nominal | Bernoulli condicional | 1 se clicou em comunicado interno |
| `clicou_link_operacional` | binária | nominal | Bernoulli condicional | 1 se clicou em link operacional |
| `sucesso_encontrou_o_que_queria` | binária | nominal | Bernoulli condicional | 1 se encontrou o que buscava |
| `tipo_conteudo_principal` | categórica | nominal | derivada do objetivo | Categoria do conteúdo mais acessado |
| `numero_interacoes` | quantitativa discreta | razão | soma de ações | Total de ações realizadas |
| `proporcao_tempo_leitura` | quantitativa contínua | razão | razão de tempos | Fração do tempo gasta lendo |
| `abandono_rapido` | binária | nominal | regra derivada | 1 se sessão foi curta e sem sucesso |

### Mini-exercício 4.1 — Propondo uma nova variável

Escolha uma situação do problema e proponha uma nova variável para o schema.

<details>
<summary>Possível direção de resposta</summary>

O que esperamos avaliar aqui não é a criatividade da variável isoladamente, mas se você consegue justificar:

1. o nome da variável;
2. seu tipo de dado;
3. seu nível de mensuração;
4. se ela deve ser gerada diretamente ou derivada de outras variáveis.

Uma boa resposta mostra coerência metodológica, não apenas uma ideia interessante.

</details>

## Etapa 5 — Fundamentos: escolhendo as distribuições certas

Nesta etapa, vamos seguir a regra de um conceito por vez. Em vez de falar de todas as distribuições ao mesmo tempo, vamos olhar separadamente para três famílias úteis neste tutorial:

1. Poisson para contagens;
2. log-normal para tempos positivos e assimétricos;
3. Bernoulli para variáveis binárias.

A pergunta orientadora é sempre a mesma:

> Que tipo de comportamento esta variável representa e que forma estatística parece compatível com isso?

### 5.1 — Distribuição de Poisson: contagens discretas

Use Poisson quando a variável representa uma contagem de eventos em um intervalo.

Abordagem incorreta comum:
- usar distribuição uniforme para pageviews só porque ela é simples.

Problema:
- a distribuição uniforme trata todos os valores como igualmente prováveis;
- isso ignora que sessões curtas são mais comuns que sessões muito longas.

Objetivo deste bloco de código:
- visualizar como diferentes valores de lambda alteram uma distribuição de contagem.

Entrada:
- três valores de `lambda`.

Saída esperada:
- histogramas mostrando que a média esperada de pageviews muda com o objetivo da sessão.

In [5]:
# Demonstração: como lambda afeta a forma da distribuição de Poisson
cenarios_poisson = [
    ('busca_pessoa', 2.0),
    ('ler_comunicado', 3.5),
    ('acompanhar_noticias', 6.0),
]

linhas_poisson = []
for cenario, lam in cenarios_poisson:
    amostras = np.random.poisson(lam, 2000)
    for valor in amostras:
        linhas_poisson.append({'cenario': cenario, 'lambda': lam, 'pageviews': int(valor)})

df_poisson = pd.DataFrame(linhas_poisson)
df_poisson['painel'] = df_poisson.apply(lambda r: f"{r['cenario']} | lambda={r['lambda']}", axis=1)

fig = px.histogram(
    df_poisson,
    x='pageviews',
    facet_col='painel',
    facet_col_wrap=3,
    histnorm='probability density',
    nbins=18,
    color='painel',
    opacity=0.75,
    title='Distribuição de Poisson para diferentes objetivos de sessão',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Pageviews')
fig.update_yaxes(title_text='Densidade')
fig

### 5.2 — Distribuição Log-Normal: tempos e durações

Use log-normal quando a variável é **sempre positiva** e tem uma cauda direita (alguns valores muito altos, mas maioria concentrada abaixo da média).

**Por que log-normal para tempos?** Porque tempos de navegação são assimétricos: a maioria das sessões é curta, mas algumas duram muito mais que o esperado.

Objetivo deste bloco de código:
- comparar a forma de distribuições log-normais para tempos de navegação.

Entrada:
- parâmetros de média logarítmica e dispersão.

Saída esperada:
- duas distribuições assimétricas, com cauda à direita e média maior que a mediana.

In [6]:
# Demonstração: por que log-normal é adequada para tempos positivos e assimétricos
amostras = []
for cenario, mu, sigma in [
    ('Tempo de busca - objetivo operacional', 4.0, 0.55),
    ('Tempo de leitura - ler comunicado', 4.3, 0.55),
]:
    valores = np.random.lognormal(mean=mu, sigma=sigma, size=3000)
    for valor in valores:
        amostras.append({'cenario': cenario, 'tempo_segundos': float(valor)})

df_lognormal = pd.DataFrame(amostras)

fig = px.histogram(
    df_lognormal,
    x='tempo_segundos',
    facet_col='cenario',
    histnorm='probability density',
    nbins=60,
    color='cenario',
    opacity=0.75,
    title='Log-normal capta a assimetria típica de tempos de navegação',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Tempo (segundos)')
fig.update_yaxes(title_text='Densidade')
fig

### 5.3 — Distribuição de Bernoulli: variáveis binárias

Use Bernoulli quando a variável assume apenas **dois valores** (0 ou 1).

**Parâmetro:** p = probabilidade de sucesso (valor 1).

> Exemplo: `usou_busca`, `clicou_comunicado`, `sucesso_encontrou_o_que_queria`.

O poder real está em fazer **p depender de outras variáveis** — por exemplo, a probabilidade de sucesso é menor se o usuário usou a busca (indica que não achou direto).

Objetivo deste bloco de código:
- mostrar probabilidades de variáveis binárias no modelo.

Entrada:
- probabilidades associadas a alguns comportamentos.

Saída esperada:
- um gráfico simples que ajude a interpretar o papel de `p` em uma variável Bernoulli.

In [7]:
probabilidades = {
    'usou_busca (buscar_doc)': 0.65,
    'clicou_comunicado (ler_comunicado)': 0.78,
    'sucesso (buscar_sistema, sem busca)': 0.76,
    'sucesso (buscar_sistema, com busca)': 0.69,
}

df_prob = pd.DataFrame(
    {'variavel': list(probabilidades.keys()), 'probabilidade': list(probabilidades.values())}
)

fig = px.bar(
    df_prob,
    x='probabilidade',
    y='variavel',
    orientation='h',
    text='probabilidade',
    range_x=[0, 1.05],
    title='Probabilidades das variáveis Bernoulli no modelo',
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_title='', xaxis_title='Probabilidade de valor = 1')
fig

### Mini-exercício 5.1 — Qual distribuição usar?

Para cada variável proposta, tente escolher a família de distribuição mais adequada.

<details>
<summary>Possível direção de resposta</summary>

Uma direção plausível seria:

- número de retornos à home: Poisson ou outra distribuição de contagem;
- duração entre cliques: log-normal, por ser positiva e assimétrica;
- sessão mobile ou não: Bernoulli, por ser binária;
- número de buscas internas: Poisson, por ser contagem.

Mais importante do que acertar o nome é justificar o tipo de fenômeno representado.

</details>

## Etapa 6 — Construindo as regras de geração

Agora saímos da escolha isolada de distribuições e passamos a combinar regras.

Estratégia desta etapa:

1. declarar parâmetros base;
2. explicitar hipóteses condicionais;
3. encapsular essas hipóteses em funções reutilizáveis.

Esse passo intermediário é importante porque evita o salto direto do conceito para um loop grande demais.

Objetivo deste bloco de código:
- definir parâmetros estruturais do modelo.

Entrada:
- sinais reais do parceiro;
- suposições calibradas com o contexto.

Saída esperada:
- listas e pesos básicos para perfis, dispositivos, dias e faixas horárias.

In [8]:
# Parâmetros-base do modelo
perfis = ['Pesquisador', 'Administrativo', 'Tecnico', 'Operacional']
pesos_perfil = [0.30, 0.27, 0.25, 0.18]

tipos_dispositivo = ['desktop', 'mobile_web', 'mobile_app']
pesos_dispositivo = [0.974, 0.014, 0.012]

dias_semana = ['segunda', 'terca', 'quarta', 'quinta', 'sexta', 'sabado', 'domingo']
pesos_dia = [0.19, 0.18, 0.20, 0.17, 0.16, 0.05, 0.05]

faixas_horarias = ['06-08', '08-11', '11-14', '14-17', '17-20', '20-06']
pesos_faixa = [0.12, 0.41, 0.17, 0.18, 0.08, 0.04]

print('Parâmetros-base carregados.')
print(f"Soma pesos perfil: {sum(pesos_perfil):.2f}")
print(f"Soma pesos dia:    {sum(pesos_dia):.2f}")
print(f"Soma pesos faixa:  {sum(pesos_faixa):.2f}")

Parâmetros-base carregados.
Soma pesos perfil: 1.00
Soma pesos dia:    1.00
Soma pesos faixa:  1.00


Checagem opcional:

O bloco abaixo apenas imprime os pesos declarados para facilitar inspeção rápida antes de seguir.

In [9]:
print('Checagem de pesos declarados:')
for nome, opcoes, pesos in [
    ('Faixas horárias', faixas_horarias, pesos_faixa),
    ('Dias da semana',  dias_semana,     pesos_dia),
]:
    print(f"\n  {nome}:")
    for op, p in zip(opcoes, pesos):
        barra = '█' * int(p * 40)
        print(f"    {op:<10} {barra} {p:.0%}")

Checagem de pesos declarados:

  Faixas horárias:
    06-08      ████ 12%
    08-11      ████████████████ 41%
    11-14      ██████ 17%
    14-17      ███████ 18%
    17-20      ███ 8%
    20-06      █ 4%

  Dias da semana:
    segunda    ███████ 19%
    terca      ███████ 18%
    quarta     ████████ 20%
    quinta     ██████ 17%
    sexta      ██████ 16%
    sabado     ██ 5%
    domingo    ██ 5%


### 6.2 — Regras condicionais: objetivos por perfil

Aqui a modelagem deixa de ser apenas marginal e passa a ser relacional.

Abordagem incorreta comum:
- assumir que todos os perfis têm a mesma distribuição de objetivos.

Problema:
- isso ignora exatamente a dimensão comportamental que queremos explorar.

In [10]:
objetivos_por_perfil = {
    'Pesquisador': {
        'buscar_documento': 0.34,
        'ler_comunicado': 0.22,
        'buscar_sistema': 0.18,
        'buscar_pessoa': 0.08,
        'acompanhar_noticias': 0.18,
    },
    'Administrativo': {
        'buscar_sistema': 0.36,
        'buscar_documento': 0.20,
        'ler_comunicado': 0.20,
        'buscar_pessoa': 0.10,
        'acompanhar_noticias': 0.14,
    },
    'Tecnico': {
        'buscar_documento': 0.28,
        'buscar_sistema': 0.30,
        'ler_comunicado': 0.14,
        'buscar_pessoa': 0.10,
        'acompanhar_noticias': 0.18,
    },
    'Operacional': {
        'buscar_sistema': 0.26,
        'ler_comunicado': 0.26,
        'acompanhar_noticias': 0.22,
        'buscar_documento': 0.14,
        'buscar_pessoa': 0.12,
    },
}

print('Hipóteses de objetivos por perfil carregadas.')

Hipóteses de objetivos por perfil carregadas.


Checagem opcional:

O bloco abaixo imprime as hipóteses por perfil para verificar se as diferenças estão legíveis antes da geração da base.

In [11]:
print('Checagem de objetivos por perfil:')
for perfil, dist in objetivos_por_perfil.items():
    print(f"\n  {perfil}:")
    for obj, p in sorted(dist.items(), key=lambda x: -x[1]):
        barra = '█' * int(p * 40)
        print(f"    {obj:<25} {barra} {p:.0%}")

Checagem de objetivos por perfil:

  Pesquisador:
    buscar_documento          █████████████ 34%
    ler_comunicado            ████████ 22%
    buscar_sistema            ███████ 18%
    acompanhar_noticias       ███████ 18%
    buscar_pessoa             ███ 8%

  Administrativo:
    buscar_sistema            ██████████████ 36%
    buscar_documento          ████████ 20%
    ler_comunicado            ████████ 20%
    acompanhar_noticias       █████ 14%
    buscar_pessoa             ████ 10%

  Tecnico:
    buscar_sistema            ████████████ 30%
    buscar_documento          ███████████ 28%
    acompanhar_noticias       ███████ 18%
    ler_comunicado            █████ 14%
    buscar_pessoa             ████ 10%

  Operacional:
    buscar_sistema            ██████████ 26%
    ler_comunicado            ██████████ 26%
    acompanhar_noticias       ████████ 22%
    buscar_documento          █████ 14%
    buscar_pessoa             ████ 12%


### 6.3 — Funções geradoras com dependências

Agora vamos encapsular as regras em funções. Leia cada função com uma pergunta em mente:

- o que veio de evidência?
- o que veio de hipótese?
- como a função transforma isso em comportamento observável?

Objetivo deste bloco de código:
- transformar regras de geração em funções reutilizáveis;
- preparar uma função principal que gere a base completa.

Entrada:
- parâmetros base;
- dicionário de objetivos por perfil.

Saída esperada:
- funções auxiliares e uma função `gerar_base(...)` pronta para reuso nos exercícios.

In [12]:
tipo_conteudo_por_objetivo = {
    'buscar_sistema': 'Site Page',
    'buscar_documento': 'Document',
    'buscar_pessoa': 'Site Page',
    'ler_comunicado': 'News Post',
    'acompanhar_noticias': 'News Post',
}

lambdas_pageviews = {
    'buscar_sistema': 3.2,
    'buscar_documento': 2.6,
    'buscar_pessoa': 2.0,
    'ler_comunicado': 3.7,
    'acompanhar_noticias': 4.2,
}

origens = ['home_direta', 'busca_interna', 'link_email', 'atalho_sistema']


def escolher_com_pesos(opcoes, pesos):
    return np.random.choice(opcoes, p=pesos)


def escolher_objetivo(perfil):
    dist = objetivos_por_perfil[perfil]
    return escolher_com_pesos(list(dist.keys()), list(dist.values()))


def escolher_origem(objetivo, faixa_horaria):
    mapa_pesos = {
        'buscar_sistema': [0.25, 0.20, 0.05, 0.50],
        'buscar_documento': [0.24, 0.44, 0.10, 0.22],
        'ler_comunicado': [0.38, 0.10, 0.32, 0.20],
        'acompanhar_noticias': [0.46, 0.08, 0.28, 0.18],
    }
    pesos = mapa_pesos.get(objetivo, [0.28, 0.32, 0.10, 0.30])
    origem = escolher_com_pesos(origens, pesos)

    if faixa_horaria == '08-11' and objetivo == 'buscar_sistema' and np.random.rand() < 0.25:
        origem = 'atalho_sistema'
    return origem


def gerar_pageviews(objetivo, tipo_dispositivo):
    lam = lambdas_pageviews[objetivo]
    if tipo_dispositivo != 'desktop':
        lam = max(1.5, lam - 0.6)
    return max(1, np.random.poisson(lam))


def gerar_tempos(objetivo, usou_busca, faixa_horaria):
    params = {
        'buscar_sistema': (4.0, 0.55, 3.2, 0.60),
        'buscar_documento': (4.0, 0.55, 3.2, 0.60),
        'buscar_pessoa': (4.0, 0.55, 3.2, 0.60),
        'ler_comunicado': (2.9, 0.45, 4.3, 0.55),
        'acompanhar_noticias': (3.2, 0.50, 4.0, 0.60),
    }
    mu_b, sig_b, mu_l, sig_l = params.get(objetivo, (3.5, 0.50, 3.8, 0.55))

    tempo_busca = np.random.lognormal(mu_b, sig_b)
    tempo_leitura = np.random.lognormal(mu_l, sig_l)

    if not usou_busca:
        tempo_busca *= 0.45

    if faixa_horaria == '08-11':
        tempo_leitura *= 0.88
        tempo_busca *= 1.08

    ruido = np.random.gamma(shape=2.0, scale=8.0)
    tempo_total = tempo_busca + tempo_leitura + ruido

    return tempo_busca, tempo_leitura, tempo_total


def gerar_flags(objetivo, origem_acesso):
    usou_busca = int(
        origem_acesso == 'busca_interna'
        or (objetivo in {'buscar_documento', 'buscar_pessoa'} and np.random.rand() < 0.45)
    )
    clicou_comunicado = int(objetivo in {'ler_comunicado', 'acompanhar_noticias'} and np.random.rand() < 0.78)
    clicou_link_operacional = int(objetivo in {'buscar_sistema', 'buscar_documento'} and np.random.rand() < 0.72)
    return usou_busca, clicou_comunicado, clicou_link_operacional


def gerar_sucesso(objetivo, usou_busca, tipo_dispositivo):
    probs_base = {
        'buscar_sistema': 0.76,
        'buscar_documento': 0.68,
        'buscar_pessoa': 0.72,
        'ler_comunicado': 0.81,
        'acompanhar_noticias': 0.79,
    }
    p = probs_base[objetivo]
    if usou_busca:
        p -= 0.07
    if tipo_dispositivo != 'desktop':
        p -= 0.04
    p = min(max(p, 0.15), 0.95)
    return int(np.random.rand() < p)


def gerar_base(n_sessoes=5000, pesos_perfil_override=None, gerar_tempos_fn=None):
    pesos_p = pesos_perfil_override if pesos_perfil_override is not None else pesos_perfil
    tempos_fn = gerar_tempos_fn if gerar_tempos_fn is not None else gerar_tempos
    linhas = []

    for i in range(1, n_sessoes + 1):
        perfil = escolher_com_pesos(perfis, pesos_p)
        dispositivo = escolher_com_pesos(tipos_dispositivo, pesos_dispositivo)
        dia = escolher_com_pesos(dias_semana, pesos_dia)
        faixa = escolher_com_pesos(faixas_horarias, pesos_faixa)

        objetivo = escolher_objetivo(perfil)
        origem = escolher_origem(objetivo, faixa)

        usou_busca, clicou_comunicado, clicou_link_op = gerar_flags(objetivo, origem)
        pageviews = gerar_pageviews(objetivo, dispositivo)

        try:
            tempo_busca, tempo_leitura, tempo_total = tempos_fn(objetivo, usou_busca, faixa, dia)
        except TypeError:
            tempo_busca, tempo_leitura, tempo_total = tempos_fn(objetivo, usou_busca, faixa)

        sucesso = gerar_sucesso(objetivo, usou_busca, dispositivo)

        tipo_conteudo = tipo_conteudo_por_objetivo[objetivo]
        n_interacoes = int(pageviews + clicou_comunicado + clicou_link_op + usou_busca)
        prop_leitura = round(tempo_leitura / tempo_total, 4) if tempo_total > 0 else 0
        abandono = int((tempo_total < 35 and pageviews <= 2) or (sucesso == 0 and tempo_total < 45))

        linhas.append({
            'session_id': i,
            'perfil_usuario': perfil,
            'tipo_dispositivo': dispositivo,
            'dia_semana': dia,
            'faixa_horaria': faixa,
            'origem_acesso': origem,
            'objetivo_sessao': objetivo,
            'pageviews': pageviews,
            'tempo_busca_segundos': round(tempo_busca, 2),
            'tempo_leitura_segundos': round(tempo_leitura, 2),
            'tempo_total_segundos': round(tempo_total, 2),
            'usou_busca': usou_busca,
            'clicou_comunicado': clicou_comunicado,
            'clicou_link_operacional': clicou_link_op,
            'sucesso_encontrou_o_que_queria': sucesso,
            'tipo_conteudo_principal': tipo_conteudo,
            'numero_interacoes': n_interacoes,
            'proporcao_tempo_leitura': prop_leitura,
            'abandono_rapido': abandono,
        })

    return pd.DataFrame(linhas)

print('Funções de geração carregadas.')
print('A função gerar_base(...) já está pronta para ser reutilizada nos exercícios.')

Funções de geração carregadas.
A função gerar_base(...) já está pronta para ser reutilizada nos exercícios.


### Mini-exercício 6.1 — Inspecionando uma função

Execute a célula abaixo e interprete as probabilidades de sucesso.

<details>
<summary>Possível direção de resposta</summary>

Esperamos que você perceba que:

- objetivos operacionais, como `buscar_documento`, tendem a ter menor probabilidade de sucesso;
- usar busca reduz um pouco a chance de sucesso, porque sinaliza dificuldade de navegação direta;
- mobile adiciona uma penalidade pequena adicional.

A resposta importante aqui é a relação entre hipótese comportamental e parâmetro probabilístico.

</details>

In [13]:
# Inspecionando as probabilidades de sucesso simuladas
print('Probabilidades de sucesso por objetivo e condição:')
print('-' * 60)
print(f"{'Objetivo':<25} {'Sem busca':<15} {'Com busca':<15} {'Com busca + mobile'}")
print('-' * 60)

probs_base = {
    'buscar_sistema':    0.76,
    'buscar_documento':  0.68,
    'buscar_pessoa':     0.72,
    'ler_comunicado':    0.81,
    'acompanhar_noticias': 0.79,
}
for obj, p in probs_base.items():
    p_sem   = p
    p_com   = max(p - 0.07, 0.15)
    p_mob   = max(p - 0.07 - 0.04, 0.15)
    print(f"  {obj:<23} {p_sem:.2f}          {p_com:.2f}          {p_mob:.2f}")


Probabilidades de sucesso por objetivo e condição:
------------------------------------------------------------
Objetivo                  Sem busca       Com busca       Com busca + mobile
------------------------------------------------------------
  buscar_sistema          0.76          0.69          0.65
  buscar_documento        0.68          0.61          0.57
  buscar_pessoa           0.72          0.65          0.61
  ler_comunicado          0.81          0.74          0.70
  acompanhar_noticias     0.79          0.72          0.68


## Etapa 7 — Gerando a base sintética

Agora montamos o loop principal. Cada iteração gera uma sessão completa, combinando todas as funções que construímos.

>  **Para entender o loop:** tente mentalmente "simular" uma sessão para um usuário "Pesquisador" às 9h de uma quarta. Qual seria seu objetivo provável? Por onde ele acessaria? Quanto tempo ele ficaria lendo vs. buscando?

Objetivo deste bloco de código:
- gerar a primeira versão da base sintética completa.

Entrada:
- função `gerar_base(...)`;
- número de sessões definido nas decisões metodológicas.

Saída esperada:
- um DataFrame com 5000 linhas e 19 variáveis.

In [14]:
n_sessoes = decisoes_metodologicas['n_sessoes']
df = gerar_base(n_sessoes=n_sessoes)
print(f"Base gerada com {len(df)} sessões e {len(df.columns)} variáveis.")
df.head(5)

Base gerada com 5000 sessões e 19 variáveis.


,session_id,perfil_usuario,tipo_dispositivo,dia_semana,faixa_horaria,origem_acesso,objetivo_sessao,pageviews,tempo_busca_segundos,tempo_leitura_segundos,tempo_total_segundos,usou_busca,clicou_comunicado,clicou_link_operacional,sucesso_encontrou_o_que_queria,tipo_conteudo_principal,numero_interacoes,proporcao_tempo_leitura,abandono_rapido
0,1,Tecnico,desktop,sexta,06-08,atalho_sistema,ler_comunicado,2,10.62,224.46,239.68,0,1,0,0,News Post,3,0.9365,0
1,2,Tecnico,desktop,quarta,06-08,home_direta,buscar_sistema,2,39.48,30.84,92.62,0,0,0,1,Site Page,2,0.3330,0
2,3,Operacional,desktop,sexta,14-17,atalho_sistema,buscar_documento,1,18.85,24.56,69.05,0,0,1,0,Document,2,0.3556,0
3,4,Pesquisador,desktop,terca,08-11,home_direta,ler_comunicado,5,8.58,29.07,56.09,0,1,0,1,News Post,6,0.5182,0
4,5,Pesquisador,desktop,sexta,08-11,atalho_sistema,buscar_sistema,2,64.54,30.51,121.39,0,0,1,1,Site Page,3,0.2514,0


## Etapa 8 — Inserindo outliers realistas

Uma base sintética sem outliers não é realista. Em comportamento digital, valores extremos existem e têm significado:

- Sessões muito longas → usuário engajado, ou perdido, ou deixou a aba aberta
- Muitas pageviews → exploração intensa de conteúdo

>  **Cuidado metodológico:** outliers sintéticos devem ter uma *justificativa comportamental*, não serem apenas "ruído aleatório".

Objetivo deste bloco de código:
- inserir casos extremos plausíveis na base.

Entrada:
- base sintética já gerada.

Saída esperada:
- uma base levemente enriquecida com outliers justificáveis em tempo e pageviews.

In [15]:
# Outliers de tempo - sessões de leitura muito intensa (~0.6% da base)
idx_tempo = df.sample(frac=0.006, random_state=42).index
fator_tempo = np.random.uniform(2.2, 4.0, len(idx_tempo))
df.loc[idx_tempo, 'tempo_leitura_segundos'] *= fator_tempo
df.loc[idx_tempo, 'tempo_total_segundos'] = (
    df.loc[idx_tempo, 'tempo_busca_segundos']
    + df.loc[idx_tempo, 'tempo_leitura_segundos']
    + np.random.uniform(10, 35, len(idx_tempo))
)

# Outliers de pageviews - navegação muito intensa (~0.4% da base)
idx_pv = df.sample(frac=0.004, random_state=7).index
df.loc[idx_pv, 'pageviews'] += np.random.randint(8, 18, len(idx_pv))
df.loc[idx_pv, 'numero_interacoes'] = (
    df.loc[idx_pv, 'pageviews']
    + df.loc[idx_pv, 'clicou_comunicado']
    + df.loc[idx_pv, 'clicou_link_operacional']
    + df.loc[idx_pv, 'usou_busca']
)

df['proporcao_tempo_leitura'] = (df['tempo_leitura_segundos'] / df['tempo_total_segundos']).round(4)
df['abandono_rapido'] = (
    ((df['tempo_total_segundos'] < 35) & (df['pageviews'] <= 2)) |
    ((df['sucesso_encontrou_o_que_queria'] == 0) & (df['tempo_total_segundos'] < 45))
).astype(int)

df_outliers = pd.concat([
    df[['tempo_total_segundos']].assign(metrica='tempo_total_segundos', valor=df['tempo_total_segundos']),
    df[['pageviews']].assign(metrica='pageviews', valor=df['pageviews']),
], ignore_index=True)

fig = px.histogram(
    df_outliers,
    x='valor',
    facet_col='metrica',
    nbins=60,
    color='metrica',
    opacity=0.75,
    title='Distribuições após inserção de outliers plausíveis',
)
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text='Valor')
fig.update_yaxes(title_text='Contagem')
fig

## Etapa 9 — Validação e exploração da base

Gerar dados é só metade do trabalho. A outra metade é **verificar se eles fazem sentido**.

> A pergunta não é "ficou perfeito?", mas sim:  
> **"A base é coerente com o caso e útil para análise?"**

### 9.1 — Checagem rápida de sanidade

Objetivo deste bloco de código:
- comparar sinais da base gerada com algumas âncoras esperadas.

Entrada:
- DataFrame sintético final.

Saída esperada:
- uma checagem rápida de sanidade, sem pretensão de validação definitiva.

In [16]:
# Comparando os sinais reais com o que geramos
print('=' * 65)
print('CHECAGEM: sinais reais vs. base sintética gerada')
print('=' * 65)

checagens = [
    ('% desktop', 0.974, (df['tipo_dispositivo'] == 'desktop').mean()),
    ('% faixa 08-11h', 0.41, (df['faixa_horaria'] == '08-11').mean()),
    ('% dia útil', 0.90, df['dia_semana'].isin(['segunda', 'terca', 'quarta', 'quinta', 'sexta']).mean()),
    ('pageviews médios', 3.2, df['pageviews'].mean()),
    ('% abandono rápido', '~', df['abandono_rapido'].mean()),
]

for nome, esperado, obtido in checagens:
    if esperado == '~':
        status = 'ok'
    else:
        status = 'ok' if abs(float(str(esperado)) - obtido) < 0.08 else 'revisar'
    print(f"  [{status}] {nome:<25} esperado: {str(esperado):<8} obtido: {obtido:.3f}")


CHECAGEM: sinais reais vs. base sintética gerada
  [ok] % desktop                 esperado: 0.974    obtido: 0.977
  [ok] % faixa 08-11h            esperado: 0.41     obtido: 0.416
  [ok] % dia útil                esperado: 0.9      obtido: 0.906
  [revisar] pageviews médios          esperado: 3.2      obtido: 3.286
  [ok] % abandono rápido         esperado: ~        obtido: 0.020


In [17]:
# Estatísticas descritivas completas
df.describe(include='all').T.round(3)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
session_id,5000.0,NaN,NaN,NaN,2500.5,1443.520003,1.0,1250.75,2500.5,3750.25,5000.0
perfil_usuario,5000,4,Pesquisador,1538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_dispositivo,5000,3,desktop,4887,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dia_semana,5000,7,quarta,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN
faixa_horaria,5000,6,08-11,2080,NaN,NaN,NaN,NaN,NaN,NaN,NaN
origem_acesso,5000,4,home_direta,1553,NaN,NaN,NaN,NaN,NaN,NaN,NaN
objetivo_sessao,5000,5,buscar_sistema,1315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pageviews,5000.0,NaN,NaN,NaN,3.2862,2.024823,1.0,2.0,3.0,4.0,23.0
tempo_busca_segundos,5000.0,NaN,NaN,NaN,33.16454,31.774299,1.53,11.61,22.395,44.29,324.97
tempo_leitura_segundos,5000.0,NaN,NaN,NaN,45.629514,41.122862,1.75,19.4775,33.06,57.21,702.88


### 9.2 — Visualizações exploratórias

Objetivo deste bloco de código:
- produzir algumas visualizações exploratórias iniciais.

Entrada:
- base sintética gerada.

Saída esperada:
- gráficos que já ajudem a iniciar a análise do AF01.

In [18]:
# Visualizações exploratórias principais usando subplots interativos
perfil_counts = df['perfil_usuario'].value_counts().reset_index()
perfil_counts.columns = ['perfil_usuario', 'sessoes']

leitura_por_faixa = df.groupby('faixa_horaria', as_index=False)['proporcao_tempo_leitura'].mean()

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Distribuição por perfil de usuário',
        'Distribuição de pageviews por sessão',
        'Tempo total por objetivo da sessão',
        'Proporção média de leitura por faixa horária',
    ),
)

for trace in px.bar(perfil_counts, x='perfil_usuario', y='sessoes').data:
    fig.add_trace(trace, row=1, col=1)

for trace in px.histogram(df, x='pageviews', nbins=25).data:
    fig.add_trace(trace, row=1, col=2)

for trace in px.box(df, x='objetivo_sessao', y='tempo_total_segundos').data:
    fig.add_trace(trace, row=2, col=1)

for trace in px.bar(leitura_por_faixa, x='faixa_horaria', y='proporcao_tempo_leitura').data:
    fig.add_trace(trace, row=2, col=2)

fig.update_layout(height=800, width=1100, showlegend=False, title_text='Exploração inicial da base sintética')
fig.update_xaxes(title_text='Perfil', row=1, col=1)
fig.update_yaxes(title_text='Sessões', row=1, col=1)
fig.update_xaxes(title_text='Pageviews', row=1, col=2)
fig.update_yaxes(title_text='Contagem', row=1, col=2)
fig.update_xaxes(title_text='Objetivo', row=2, col=1)
fig.update_yaxes(title_text='Segundos', row=2, col=1)
fig.update_xaxes(title_text='Faixa horária', row=2, col=2)
fig.update_yaxes(title_text='Proporção média', row=2, col=2)
fig

### 9.3 — Checagem de coerência interna

As regras condicionais que criamos deveriam se refletir nos dados. Vamos verificar.

In [19]:
# Cruzamento: perfil × objetivo (as proporções devem refletir os pesos que definimos)
print('Proporção de objetivos por perfil (deve refletir objetivos_por_perfil):')
pd.crosstab(df['perfil_usuario'], df['objetivo_sessao'], normalize='index').round(3)


Proporção de objetivos por perfil (deve refletir objetivos_por_perfil):


objetivo_sessao,acompanhar_noticias,buscar_documento,buscar_pessoa,buscar_sistema,ler_comunicado
perfil_usuario,,,,,
Administrativo,0.140,0.195,0.098,0.363,0.204
Operacional,0.248,0.140,0.108,0.248,0.255
Pesquisador,0.173,0.343,0.085,0.163,0.237
Tecnico,0.191,0.270,0.112,0.288,0.139


In [20]:
# Cruzamento: objetivo × tempos médios
print('Tempos médios por objetivo (objetivos operacionais devem ter mais tempo de busca):')
df.groupby('objetivo_sessao')[['tempo_busca_segundos','tempo_leitura_segundos','tempo_total_segundos']].mean().round(1)


Tempos médios por objetivo (objetivos operacionais devem ter mais tempo de busca):


,tempo_busca_segundos,tempo_leitura_segundos,tempo_total_segundos
objetivo_sessao,,,
acompanhar_noticias,14.3,61.2,91.7
buscar_documento,55.5,27.9,99.2
buscar_pessoa,50.7,27.9,93.8
buscar_sistema,36.1,27.6,79.7
ler_comunicado,10.7,84.6,111.4


In [21]:
# Taxa de sucesso por objetivo e uso de busca
print('Taxa de sucesso: quem usou busca deve ter taxa ligeiramente menor')
df.groupby(['objetivo_sessao','usou_busca'])['sucesso_encontrou_o_que_queria'].mean().unstack().round(3)


Taxa de sucesso: quem usou busca deve ter taxa ligeiramente menor


usou_busca,0,1
objetivo_sessao,,
acompanhar_noticias,0.794,0.806
buscar_documento,0.629,0.614
buscar_pessoa,0.733,0.621
buscar_sistema,0.767,0.618
ler_comunicado,0.799,0.788


### Mini-exercício 9.1 — Encontrando incoerências

Use as saídas abaixo para verificar se as hipóteses embutidas realmente aparecem nos dados.

<details>
<summary>Possível direção de resposta</summary>

Esperamos observar algo próximo de:

- `Pesquisador` com maior proporção de `buscar_documento`;
- faixa `08-11` com menor proporção média de leitura do que faixas menos operacionais;
- sessões com `usou_busca = 1` exibindo maior tempo médio de busca.

Se isso não aparecer, a modelagem precisa ser revisada. Esse tipo de incoerência não é um fracasso; é parte do processo analítico.

</details>

In [22]:
# Célula para o exercício 9.1 — execute e análise
print("1. Proporção de 'buscar_documento' por perfil:")
print(df.groupby('perfil_usuario')['objetivo_sessao'].apply(lambda x: (x == 'buscar_documento').mean()).round(3))
print()

print('2. Proporção média de tempo de leitura por faixa:')
print(df.groupby('faixa_horaria')['proporcao_tempo_leitura'].mean().sort_index().round(3))
print()

print('3. Tempo de busca médio por uso_busca:')
print(df.groupby('usou_busca')['tempo_busca_segundos'].mean().round(1))


1. Proporção de 'buscar_documento' por perfil:
perfil_usuario
Administrativo    0.195
Operacional       0.140
Pesquisador       0.343
Tecnico           0.270
Name: objetivo_sessao, dtype: float64

2. Proporção média de tempo de leitura por faixa:
faixa_horaria
06-08    0.489
08-11    0.451
11-14    0.458
14-17    0.467
17-20    0.475
20-06    0.484
Name: proporcao_tempo_leitura, dtype: float64

3. Tempo de busca médio por uso_busca:
usou_busca
0    20.4
1    60.4
Name: tempo_busca_segundos, dtype: float64


## Etapa 10 — Exercícios práticos

Os exercícios abaixo servem para consolidar raciocínio, não apenas execução mecânica.

Ao terminar cada exercício, tente responder:

1. O que mudou na base?
2. Por que mudou?
3. O que essa mudança representa no problema real do IPT?

### Exercício 1 — Criando uma nova variável derivada

Tarefa: crie uma variável `frustracao_sintetica` que marque sessões com falha e tempo de busca acima da mediana.

In [23]:
# ── Exercício 1 — complete o código ────────────────────────────────────────

mediana_busca = df['tempo_busca_segundos'].median()

# TODO: crie a coluna 'frustração_sintetica' seguindo a definição acima
# df["frustração_sintetica"] = ...

# TODO: calcule e imprima a proporção geral de frustração
# TODO: mostre a taxa de frustração por perfil

print(f"Mediana do tempo de busca: {mediana_busca:.1f} segundos")
print()
print('Complete o código acima para criar a variável de frustração.')


Mediana do tempo de busca: 22.4 segundos

Complete o código acima para criar a variável de frustração.


Critério de validação esperado:

- a coluna deve ser binária;
- a proporção total não deve ser próxima de 0 nem de 1;
- perfis mais operacionais podem apresentar taxas maiores, dependendo das suas regras.

### Exercício 2 — Alterando os pesos dos perfis

Tarefa: re-gerar a base com maior peso para o perfil Operacional.

In [24]:
# ── Exercício 2 — complete o código ────────────────────────────────────────

pesos_perfil_v2 = [0.20, 0.22, 0.22, 0.36]

# TODO: gere uma nova base df_v2 usando pesos_perfil_v2 (copie e adapte o loop da Etapa 7)
# Dica: crie uma função gerar_base(n, pesos_p) para evitar duplicar o loop

# TODO: compare df["objetivo_sessao"].value_counts(normalize=True)
#       com df_v2["objetivo_sessao"].value_counts(normalize=True)

# TODO: compare a taxa de abandono_rapido entre as duas bases

print('Implemente a re-geração aqui e compare os resultados.')


Implemente a re-geração aqui e compare os resultados.


Critério de validação esperado:

- a distribuição de objetivos deve se deslocar;
- a taxa de abandono pode mudar por efeito indireto das novas composições;
- a explicação deve rastrear o encadeamento entre perfil, objetivo e comportamento.

### Exercício 3 — Adicionando dependência temporal

Tarefa: criar uma versão de `gerar_tempos` que aumente o tempo de busca em quartas-feiras entre 08 e 11h.

In [25]:
# ── Exercício 3 — complete o código ────────────────────────────────────────

def gerar_tempos_v2(objetivo, usou_busca, faixa_horaria, dia_semana):
    '\n    Versão aprimorada de gerar_tempos com dependência dia × faixa.\n    \n    TODO: implemente a lógica adicional para quarta + 08-11h\n    '
    # Copie aqui a lógica original de gerar_tempos
    # e adicione a condição extra para quarta + 08-11h
    pass  # remova este pass ao implementar

# TODO: re-gere a base usando gerar_tempos_v2
# TODO: verifique o efeito com:
#   df_v3.groupby(["dia_semana","faixa_horaria"])["tempo_busca_segundos"].mean()

print('Implemente gerar_tempos_v2 e re-gere a base.')


Implemente gerar_tempos_v2 e re-gere a base.


Critério de validação esperado:

- a nova base deve exibir aumento localizado em `quarta` + `08-11`;
- esse efeito deve ser verificável em uma agregação por dia e faixa.

### Exercício 4 — Visualizando a tensão central do IPT

Tarefa: construir um gráfico que compare tempo de busca e tempo de leitura por perfil.

In [26]:
figura_base = df.groupby('perfil_usuario', as_index=False)[['tempo_busca_segundos', 'tempo_leitura_segundos']].mean()
figura_long = figura_base.melt(id_vars='perfil_usuario', var_name='tipo_tempo', value_name='segundos')

fig = px.bar(
    figura_long,
    x='perfil_usuario',
    y='segundos',
    color='tipo_tempo',
    barmode='group',
    title='Tensão central do IPT: tempo buscando vs. tempo lendo',
)
fig.update_xaxes(title_text='Perfil')
fig.update_yaxes(title_text='Segundos médios')
fig

Critério de validação esperado:

- o gráfico deve permitir comparação clara entre perfis;
- legenda, título e eixos precisam estar legíveis para uso posterior no AF01.

## Exportando a base

Quando estiver satisfeito com a base, descomente as linhas abaixo para salvar.

In [27]:
# Descomente para exportar
# df.to_csv("dataset_sintetico_ipt_v1.csv", index=False)
# print(f"Base exportada: {len(df)} sessões × {len(df.columns)} variáveis")
# df.head()


## Resumo: o que este tutorial cobriu

| O que aprendemos | Como se aplica no AF01 |
|-----------------|------------------------|
| Separar evidência de hipótese | Justificar cada escolha de geração |
| Definir a unidade de análise | Fundamentar a granularidade do dataset |
| Montar um schema documentado | Base para classificar variáveis |
| Escolher distribuições adequadas | Justificar Poisson, log-normal e Bernoulli |
| Criar dependências condicionais | Mostrar coerência interna da base |
| Validar com sinais reais | Comparar proporções geradas com âncoras observadas |
| Inserir outliers com justificativa | Aumentar realismo sem perder transparência |

Lembrete final:

O AF01 não deve apenas repetir este notebook. O grupo precisa justificar escolhas, criticar limitações, propor ajustes e interpretar os padrões à luz do problema do IPT.